In [1]:
import json
import os
import sys
from pathlib import Path
from typing import Any, TypeVar
from uuid import uuid4

import pydantic
from dotenv import load_dotenv
from icecream import ic
from openai import OpenAI
from tqdm import tqdm

from PydanticContracts import (
    BoundaryClarityJudgeResult,
    ChunkScoreJudgeResult,
    ContextualCoherenceJudgeResult,
    GeneralJudgeResult,
    HopeConceptUnityJudgeResult,
    HopeInformationPreservationJudgeResult,
    HopeSemanticIndependenceJudgeResult,
    IntrachunkCohesionJudgeResult,
    SizeComplianceJudgeResult,
    SyntheticChunkingExample,
)
from TokenUsage import (
    append_token_usage,
    load_token_usage,
    summarize_token_usage,
)

ChecksT = TypeVar("ChecksT", bound=pydantic.BaseModel)
ResultT = TypeVar("ResultT", bound=pydantic.BaseModel)

load_dotenv()

### Generator

MODEL_NAME = "deepseek-v4-flash"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
REASONING = False
REASONING_EFFORT = "medium"
MAX_TOKENS = (8192, 10000)[REASONING]
TIMEOUT_SECONDS = 240.0
PAIRS_PER_PROMPT = 30
REGENERATION_ATTEMPTS = 20
USE_JUDGE_FEEDBACK_ON_EVEN_ATTEMPTS = True

### Judge
JUDGE_MODEL_NAME = "deepseek-v4-pro"
JUDGE_BASE_URL = "https://api.deepseek.com"
JUDGE_TEMPERATURE = 0.0
JUDGE_REASONING = True
JUDGE_REASONING_EFFORT = "high"
JUDGE_MAX_TOKENS = (4096, 24000)[JUDGE_REASONING]
JUDGE_REGENERATION_ATTEMPTS = 20

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "prompts").is_dir() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
TOKEN_USAGE_PATH = OUTPUT_ROOT / "token_usage.jsonl"
RUN_ID = uuid4().hex
SELECTED_PROMPTS = [
    (Path("general_validation.md"), GeneralJudgeResult),
    # (Path("metrics/size_compliance.md"), SizeComplianceJudgeResult),
    (Path("metrics/intrachunk_cohesion.md"), IntrachunkCohesionJudgeResult),
    (Path("metrics/contextual_coherence.md"), ContextualCoherenceJudgeResult),
    (Path("metrics/boundary_clarity.md"), BoundaryClarityJudgeResult),
    (Path("metrics/chunk_score.md"), ChunkScoreJudgeResult),
    (Path("metrics/hope_concept_unity.md"), HopeConceptUnityJudgeResult),
    (
        Path("metrics/hope_semantic_independence.md"),
        HopeSemanticIndependenceJudgeResult,
    ),
    (
        Path("metrics/hope_information_preservation.md"),
        HopeInformationPreservationJudgeResult,
    ),
]

ic(SELECTED_PROMPTS)

client = OpenAI(
    api_key=os.environ["API_KEY"],
    base_url=BASE_URL,
    timeout=TIMEOUT_SECONDS,
)

ic| SELECTED_PROMPTS: [(PosixPath('general_validation.md'),
                        <class 'PydanticContracts.GeneralJudgeResult'>),
                       (PosixPath('metrics/intrachunk_cohesion.md'),
                        <class 'PydanticContracts.IntrachunkCohesionJudgeResult'>),
                       (PosixPath('metrics/contextual_coherence.md'),
                        <class 'PydanticContracts.ContextualCoherenceJudgeResult'>),
                       (PosixPath('metrics/boundary_clarity.md'),
                        <class 'PydanticContracts.BoundaryClarityJudgeResult'>),
                       (PosixPath('metrics/chunk_score.md'),
                        <class 'PydanticContracts.ChunkScoreJudgeResult'>),
                       (PosixPath('metrics/hope_concept_unity.md'),
                        <class 'PydanticContracts.HopeConceptUnityJudgeResult'>),
                       (PosixPath('metrics/hope_semantic_independence.md'),
                        <class 'PydanticContracts

In [3]:
def save_json(data: dict[str, Any] | list[dict[str, Any]], path: Path) -> Path:
    """Save JSON objects in a human-readable UTF-8 file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return path

In [4]:
def with_json_schema(
    prompt: str, result_model: type[pydantic.BaseModel]
) -> str:
    """Append a compact Pydantic JSON schema to a system prompt."""
    schema = json.dumps(
        result_model.model_json_schema(),
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return f"{prompt.rstrip()}\n\nJSON schema ответа:\n{schema}"

In [5]:
def llm_judge(
    example: SyntheticChunkingExample,
    system_prompt: str,
    metric_prompt: str,
    result_model: type[ResultT],
    prompt: str,
    pair_number: int,
    generator_attempt: int,
    used_judge_feedback: bool,
) -> ResultT:
    messages = [
        {
            "role": "system",
            "content": with_json_schema(system_prompt, result_model),
        },
        {
            "role": "user",
            "content": (
                f"{metric_prompt}\n\n"
                "Проверь следующий синтетический пример:\n\n"
                f"{example.model_dump_json(indent=2)}"
            ),
        },
    ]

    for judge_attempt in range(1, JUDGE_REGENERATION_ATTEMPTS + 1):
        response = client.chat.completions.create(
            model=JUDGE_MODEL_NAME,
            messages=messages,
            temperature=JUDGE_TEMPERATURE,
            max_tokens=JUDGE_MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={
                "thinking": {"type": ("disabled", "enabled")[JUDGE_REASONING]}
            },
            reasoning_effort=JUDGE_REASONING_EFFORT,
        )
        try:
            content = response.choices[0].message.content
            verdict = result_model.model_validate_json(content)
        except (
            pydantic.ValidationError,
            json.JSONDecodeError,
            IndexError,
            AttributeError,
            TypeError,
        ):
            append_token_usage(
                response,
                TOKEN_USAGE_PATH,
                run_id=RUN_ID,
                prompt=prompt,
                pair_number=pair_number,
                role="judge",
                generator_attempt=generator_attempt,
                judge_attempt=judge_attempt,
                used_judge_feedback=used_judge_feedback,
                model=JUDGE_MODEL_NAME,
                endpoint=JUDGE_BASE_URL,
                temperature=JUDGE_TEMPERATURE,
                reasoning=JUDGE_REASONING,
                reasoning_effort=JUDGE_REASONING_EFFORT,
                max_tokens=JUDGE_MAX_TOKENS,
                result="invalid_json",
            )
            print("Retrying judging..")
            continue

        append_token_usage(
            response,
            TOKEN_USAGE_PATH,
            run_id=RUN_ID,
            prompt=prompt,
            pair_number=pair_number,
            role="judge",
            generator_attempt=generator_attempt,
            judge_attempt=judge_attempt,
            used_judge_feedback=used_judge_feedback,
            model=JUDGE_MODEL_NAME,
            endpoint=JUDGE_BASE_URL,
            temperature=JUDGE_TEMPERATURE,
            reasoning=JUDGE_REASONING,
            reasoning_effort=JUDGE_REASONING_EFFORT,
            max_tokens=JUDGE_MAX_TOKENS,
            result=("accepted" if verdict.valid else "judge_rejected"),
        )
        return verdict

    raise RuntimeError(
        f"Judge did not return valid {result_model.__name__} JSON after "
        f"{JUDGE_REGENERATION_ATTEMPTS} attempts"
    )

In [6]:
def generate(
    system_prompt: str,
    judge_system_prompt: str,
    user_prompt: str,
    judge_metric_prompt: str,
    judge_feedback_prompt: str,
    judge_result_model: type[ResultT],
    prompt: str,
    pair_number: int,
):
    last_rejected_example = None
    last_judge_verdict = None

    def log_generator_response(response: Any, result: str) -> None:
        append_token_usage(
            response,
            TOKEN_USAGE_PATH,
            run_id=RUN_ID,
            prompt=prompt,
            pair_number=pair_number,
            role="generator",
            generator_attempt=attempt,
            judge_attempt=None,
            used_judge_feedback=used_judge_feedback,
            model=MODEL_NAME,
            endpoint=BASE_URL,
            temperature=TEMPERATURE,
            reasoning=REASONING,
            reasoning_effort=REASONING_EFFORT,
            max_tokens=MAX_TOKENS,
            result=result,
        )

    for attempt in range(1, REGENERATION_ATTEMPTS + 1):
        messages = [
            {
                "role": "system",
                "content": with_json_schema(
                    system_prompt, SyntheticChunkingExample
                ),
            },
            {"role": "user", "content": user_prompt},
        ]
        used_judge_feedback = (
            USE_JUDGE_FEEDBACK_ON_EVEN_ATTEMPTS
            and attempt % 2 == 0
            and last_rejected_example is not None
            and last_judge_verdict is not None
        )
        if used_judge_feedback:
            messages.extend(
                [
                    {
                        "role": "assistant",
                        "content": last_rejected_example.model_dump_json(indent=2),
                    },
                    {
                        "role": "user",
                        "content": (
                            f"{judge_feedback_prompt.rstrip()}\n\n"
                            "Полный verdict судьи:\n"
                            f"{last_judge_verdict.model_dump_json(indent=2)}"
                        ),
                    },
                ]
            )

        last_rejected_example = None
        last_judge_verdict = None

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": ("disabled", "enabled")[REASONING]}},
            reasoning_effort=REASONING_EFFORT,
        )
        try:
            content = response.choices[0].message.content
            result = SyntheticChunkingExample.model_validate_json(content)
        except (
            pydantic.ValidationError,
            json.JSONDecodeError,
            IndexError,
            AttributeError,
            TypeError,
        ):
            log_generator_response(response, "invalid_json")
            tqdm.write("Retrying..")
            continue

        print("Sending to judge..")
        try:
            judge_verdict = llm_judge(
                example=result,
                system_prompt=judge_system_prompt,
                metric_prompt=judge_metric_prompt,
                result_model=judge_result_model,
                prompt=prompt,
                pair_number=pair_number,
                generator_attempt=attempt,
                used_judge_feedback=used_judge_feedback,
            )
        except Exception:
            log_generator_response(response, "judge_error")
            raise

        if not judge_verdict.valid:
            log_generator_response(response, "judge_rejected")
            tqdm.write("Judge declined, retrying..")
            ic(judge_verdict)
            last_rejected_example = result
            last_judge_verdict = judge_verdict
            continue

        log_generator_response(response, "accepted")
        print("Judge accepted")
        return result.model_dump()

    raise RuntimeError(
        f"Generator did not produce a judge-approved "
        f"{judge_result_model.__name__} example after "
        f"{REGENERATION_ATTEMPTS} attempts"
    )

In [7]:
system_prompt = (PROMPTS_ROOT / "system.md").read_text(encoding="utf-8")
judge_system_prompt = (PROMPTS_ROOT / "judge" / "system.md").read_text(encoding="utf-8")
judge_feedback_prompt = (PROMPTS_ROOT / "judge_feedback.md").read_text(encoding="utf-8")

for prompt_path, judge_result_model in tqdm(
    SELECTED_PROMPTS, desc="Prompts", position=0
):
    prompt_name = prompt_path.stem
    user_prompt = (PROMPTS_ROOT / prompt_path).read_text(encoding="utf-8")
    judge_metric_prompt = (PROMPTS_ROOT / "judge" / prompt_path).read_text(
        encoding="utf-8"
    )
    results = []
    output_path = ""
    for pair_number in tqdm(
        range(1, PAIRS_PER_PROMPT + 1), desc="Items", position=1, leave=False
    ):
        result = generate(
            system_prompt=system_prompt,
            judge_system_prompt=judge_system_prompt,
            user_prompt=user_prompt,
            judge_metric_prompt=judge_metric_prompt,
            judge_feedback_prompt=judge_feedback_prompt,
            judge_result_model=judge_result_model,
            prompt=str(prompt_path),
            pair_number=pair_number,
        )

        results.append(result)

        output_path = save_json(results, OUTPUT_ROOT / f"{prompt_name}.json")
    tqdm.write(f"Saved {output_path}")

Prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Sending to judge..


                                              
Prompts:   0%|          | 0/8 [01:56<?, ?it/s]ic| judge_verdict: GeneralJudgeResult(valid=False, quality_score=1, issues=[JudgeIssue(severity='major', code='unreported_boundary_merges', message='Negative содержит дополнительные не заявленные изменения границ: объединение раздела 2 (тело) с разделом 3, а также раздела 4 с разделом 5.'), JudgeIssue(severity='major', code='controlled_change_inaccurate', message='controlled_change и contrast_rationale упоминают только две ошибки, хотя фактически изменений больше, поэтому описание не соответствует реальным изменениям.')], checks=GeneralChecks(positive_chunks_logically_complete=True, positive_contextually_clear=True, negative_has_multiple_controlled_errors=True, negative_error_count_valid=False, size_quality_degraded=True, intrachunk_cohesion_degraded=True, contextual_coherence_degraded=True, boundary_clarity_degraded=True, information_preservation_degraded=True, at_least_two_target_properties_d

Judge declined, retrying..
Sending to judge..


Judge accepted
Sending to judge..


...

                                                             
Prompts:  75%|███████▌  | 6/8 [13:28:25<3:44:35, 6737.55s/it]ic| judge_verdict: HopeSemanticIndependenceJudgeResult(valid=False, quality_score=1, issues=[JudgeIssue(severity='fatal', code='text_modified', message="В negative отсутствует пункт '1.3.' перед 'Место нахождения...', что нарушает требование сохранять исходный текст и границы изменения."), JudgeIssue(severity='fatal', code='cue_question_invalid', message="Вопрос не порождает семантическую зависимость: отделённый чанк с исключением самостоятельно отвечает 'сделки в процессе обычной хозяйственной деятельности'; чанк с определением крупной сделки не становится неоднозначным или зависимым."), JudgeIssue(severity='minor', code='unrelated_split', message='Дополнительное разделение уставного капитала и долей не связано с cue_question и добавляет постороннее изменение.')], checks=HopeSemanticIndependenceChecks(same_source_text=False, boundary_only_change=False, cue_questio

Judge declined, retrying..
Sending to judge..


                                                             
Prompts:  75%|███████▌  | 6/8 [13:29:31<4:29:50, 8095.18s/it]

Judge declined, retrying..


RuntimeError: Generator did not produce a judge-approved HopeSemanticIndependenceJudgeResult example after 20 attempts

In [ ]:
usage_df = load_token_usage(TOKEN_USAGE_PATH)
usage_summary = summarize_token_usage(usage_df)
usage_summary

In [ ]:
tockens_cols = [i for i in usage_summary.columns if 'token' in i]
usage_summary[['model', 'role', *tockens_cols]]

In [ ]:
a = ic(156209 + 51408)
b = ic(138700 + 67227)
a - b